<a href="https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-1/04-pronostica-sin-hacer-trampa/solucion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Solución 1.4 · Pronostica sin hacer trampa

**Serie:** Del Excel al Machine Learning · **Temporada 1** · 🎬 Video próximamente

Estas son las respuestas del ejercicio. Si la tuya es distinta pero da el mismo resultado, ¡también está bien! En programación suele haber más de una forma de llegar a la respuesta.

### Prepara los datos

Ejecuta estas tres celdas. Ya dejan listas `tabla`, `entreno` (hasta junio de 2025) y `prueba` (desde julio de 2025).

Algunos ejercicios usan variables de los anteriores, así que **ejecútalos en orden**.

In [1]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

DATOS = "https://raw.githubusercontent.com/PercyTechX/del-excel-al-ml/main/datos/"

In [2]:
def tabla_de_pronostico(ventas, clientes):
    """Una fila por cliente y mes, con cero si no compró, y lo que se sabía antes de ese mes."""
    meses = ventas["fecha"].str[:7].rename("mes")
    todos = pd.MultiIndex.from_product(
        [clientes["id_cliente"], sorted(meses.unique())], names=["id_cliente", "mes"]
    )
    compras = ventas.groupby(["id_cliente", meses])["total"].sum()
    tabla = compras.reindex(todos, fill_value=0).reset_index()

    por_cliente = tabla.groupby("id_cliente")["total"]
    tabla["mes_anterior"] = por_cliente.shift(1)
    tabla["promedio_3"] = por_cliente.transform(lambda s: s.shift(1).rolling(3).mean())
    tabla["promedio_6"] = por_cliente.transform(lambda s: s.shift(1).rolling(6).mean())
    tabla["numero_mes"] = tabla["mes"].str[5:].astype(int)

    columnas = ["id_cliente", "tipo", "ciudad", "id_vendedor", "fecha_alta"]
    tabla = pd.merge(tabla, clientes[columnas], on="id_cliente", how="left")
    tabla = tabla.dropna(subset=["promedio_6"])
    return tabla[tabla["mes"] + "-01" >= tabla["fecha_alta"]].drop(columns="fecha_alta")

In [3]:
ventas = pd.read_csv(DATOS + "ventas.csv")
clientes = pd.read_csv(DATOS + "clientes.csv")

tabla = tabla_de_pronostico(ventas, clientes)

entreno = tabla[tabla["mes"] < "2025-07"]
prueba = tabla[tabla["mes"] >= "2025-07"]

print(len(entreno), len(prueba))

3475 1793


### 1. ¿Fuga o no?

Quieres predecir el `total` de un cliente en un mes. Responde en un comentario si cada columna se puede usar, y por qué:

- a) `tipo`
- b) `promedio_3`
- c) La cantidad de presentaciones que compró ese mismo mes
- d) `numero_mes`
- e) Si ese mismo mes pagó con crédito

In [4]:
# a) Sí: el tipo de cliente se conoce desde antes.
# b) Sí: mira solo los 3 meses anteriores.
# c) No: es fuga. La cantidad de ese mes solo se conoce cuando el mes termina.
# d) Sí: siempre sabes qué mes viene.
# e) No: también es fuga. Se conoce después de la compra.

### 2. Comprueba la tabla

Filtra las filas del cliente `C0002` y muestra `mes`, `total`, `mes_anterior` y `promedio_3` de las primeras 5. Comprueba a mano: `mes_anterior` de agosto debe ser el `total` de julio, `6790.5`.

In [5]:
cliente = tabla[tabla["id_cliente"] == "C0002"]

cliente[["mes", "total", "mes_anterior", "promedio_3"]].head()

,mes,total,mes_anterior,promedio_3
30,2024-07,6790.50,7755.00,6195.650000
31,2024-08,6273.40,6790.50,6363.416667
32,2024-09,3754.60,6273.40,6939.633333
33,2024-10,6051.15,3754.60,5606.166667
34,2024-11,4877.10,6051.15,5359.716667


### 3. Tu línea base

Calcula el MAE de usar `mes_anterior` como pronóstico, en `prueba`. Compáralo con los S/ 855 del promedio de 6 meses. ¿Cuál es mejor? ¿Por qué crees que pasa? Responde en un comentario.

In [6]:
print(round(mean_absolute_error(prueba["total"], prueba["mes_anterior"])))

# Es peor que el promedio de 6 meses: un solo mes tiene mucho ruido,
# y promediar varios lo suaviza.

1032


### 4. Otro modelo

Entrena una recta con `promedio_3` y mide su MAE en prueba. Debe ser `924`. ¿Le gana a usar `promedio_3` directamente, que da `889`?

In [7]:
modelo = LinearRegression()
modelo.fit(entreno[["promedio_3"]], entreno["total"])

print(round(mean_absolute_error(prueba["total"], modelo.predict(prueba[["promedio_3"]]))))

# No le gana: 924 contra 889.

924


### 5. Dónde falla más

Calcula el error de cada fila de `prueba` usando `promedio_6`, con `(prueba["total"] - prueba["promedio_6"]).abs()`, y promédialo por `numero_mes` con `groupby`. Diciembre debe dar cerca de `1031`.

¿En qué dos meses falla más? ¿Por qué? Responde en un comentario.

> 💡 Guarda el error en una variable, no en una columna. En `groupby` puedes usar una columna de la tabla, aunque el cálculo sea otra serie: `error.groupby(prueba["numero_mes"]).mean()`.

In [8]:
error = (prueba["total"] - prueba["promedio_6"]).abs()

print(error.groupby(prueba["numero_mes"]).mean().round())

# Falla más en julio y diciembre: son los picos de Fiestas Patrias y Navidad,
# y el promedio de los meses previos no los anticipa.

numero_mes
7      961.0
8      792.0
9      784.0
10     768.0
11     793.0
12    1031.0
dtype: float64


### 6. El cliente sin compras

Filtra de `prueba` las filas con `promedio_3` igual a 0: clientes sin compras en 3 meses. Deben ser `158`. Muestra la media de su `total`. ¿Tendría sentido pronosticarles S/ 1,000? Responde en un comentario.

In [9]:
sin_compras = prueba[prueba["promedio_3"] == 0]

print(len(sin_compras))
print(round(sin_compras["total"].mean(), 2))

# No: casi no compran. Conviene tratarlos aparte.

158
7.53


## 🎉 ¡Bien hecho!

Nos vemos en el **episodio 1.5: Más variables y la temporada**.

---

*Del Excel al Machine Learning* · por [Percy Nazario](https://www.linkedin.com/in/percy-nazario/) · [Repositorio](https://github.com/PercyTechX/del-excel-al-ml) · Contenido bajo licencia CC BY 4.0